# Silver Layer Overview

This notebook implements the **Silver layer** — the second stage of the Medallion Architecture (Bronze → Silver → Gold). Where the Bronze notebook lands the raw Olist data as-is, Silver turns it into clean, standardized, analysis-ready tables:

* Each Silver table is derived from its Bronze counterpart with `spark.table(...)` — the Bronze Delta tables are the only input.
* **Cleaning and standardization** — trimming whitespace, standardizing casing and normalizing formats such as ZIP code prefixes — so that identifiers and location fields join and group consistently.
* **Derived business columns** — `is_delivered`, `delivery_days` and `item_value` are computed once here, so the Gold layer never has to re-derive them.
* **Data quality checks** — null, duplicate, uniqueness and consistency checks run against the transformed DataFrames before and around persistence.

Data quality matters at this stage because every Gold metric and every join is built on Silver: a trimmed key that does not match, a duplicate row, or an inconsistent derived column would silently distort the analytical layer.

The notebook processes the five datasets in turn — orders, order items, products, customers and sellers — each ending in a `silver_*` Delta table in `workspace.olist`, which the Gold notebook (`03_gold_analytics`) and the end-to-end validation notebook (`04_validation_optimization_sql`) build on.

In [0]:
# Dependency cell (added during notebook split).
# order_items_df and seller_raw are defined in 01_bronze_ingestion; reload them here
# from the Bronze Delta tables written by that notebook (same data, identical schemas)
# so the exploration cells below are runnable standalone.
# Note: the wildcard functions import already exists in the Silver orders section below
# (original olist_full_backup cell 14) and covers all functions used in this notebook.
order_items_df = spark.table('workspace.olist.bronze_order_items')
seller_raw = spark.table('workspace.olist.bronze_sellers')

# Silver orders

## Orders Data Transformation

The Silver orders table starts from the Bronze landing table — `spark.table('workspace.default.bronze_orders')` — loaded into `silver_orders`. Two derived columns are then added:

* **`is_delivered`** — `'yes'` when `order_status == 'delivered'`, otherwise `'no'`. This turns the free-text status column into a simple delivery flag that is easy to filter and aggregate on.
* **`delivery_days`** — `datediff(order_delivered_customer_date, order_purchase_timestamp)`, the number of days between purchase and customer delivery. It is naturally null for orders that were never delivered.

Before the table is written, the in-progress DataFrame is put through several checks:

* **Uniqueness and nulls** — duplicate `order_id` values (grouped counts greater than 1), null `order_id`, and null `customer_id`.
* **Nulls in the timestamp columns** — null `order_purchase_timestamp`; null `order_approved_at` (total count, breakdown by `order_status`, and delivered orders with a null approval timestamp); and null `order_delivered_carrier_date` by status, including delivered orders missing the carrier date or both delivery dates.
* **Consistency of the derived columns** — rows where `is_delivered = 'yes'` but `delivery_days` is null, and rows where `is_delivered = 'no'` but `delivery_days` is populated. Either combination would mean the derived columns disagree with the underlying data.

These checks distinguish legitimate nulls (orders that were cancelled or never shipped) from genuine anomalies (orders marked delivered that are missing delivery dates) before the data is persisted.

In [0]:
# Lets create silver data table from bronze_orders
silver_orders=spark.table('workspace.default.bronze_orders')

In [0]:
# creating a column in which we get yes or no based on oredr status
from pyspark.sql.functions import *
silver_orders=silver_orders.withColumn('is_delivered',when(col('order_status')=='delivered','yes').otherwise('no'))

In [0]:

silver_orders=silver_orders.withColumn('delivery_days',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [0]:
# Lets check is there any duplicate order_id
silver_orders.filter(col('order_id').isNotNull()).groupBy('order_id').count().filter(col('count')>1).display()

In [0]:
# checking nulls
silver_orders.filter(col('order_id').isNull()).display()


In [0]:
# checking nulls in customers
silver_orders.filter(col('customer_id').isNull()).display()

In [0]:
silver_orders.filter(col("order_purchase_timestamp").isNull()).count()


In [0]:
silver_orders.filter(col("order_approved_at").isNull()).count()

In [0]:
# checking those null order_approved_at rows
silver_orders.filter(
    col("order_approved_at").isNull()
).groupBy(
    "order_status"
    
).count().orderBy(
    col("count").desc()
).display()

In [0]:
silver_orders.filter((col('order_status')=='delivered') & (col('order_approved_at').isNull())).select(
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).display()


In [0]:
silver_orders.filter(col('order_delivered_carrier_date').isNull()).groupBy('order_status').count().orderBy(col('count').desc()).display()

In [0]:
silver_orders.filter((col('order_status')=='delivered') & (col('order_delivered_carrier_date').isNull())).select(
    'order_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
).display()

In [0]:
silver_orders.filter((col('order_status')=='delivered') & (col('order_delivered_carrier_date').isNull()) & (col('order_delivered_customer_date').isNull())).select(
    '*'
).display()


In [0]:
silver_orders.filter((col('is_delivered')=='yes') & (col('delivery_days').isNull())).count()

In [0]:
silver_orders.filter((col('is_delivered')=='no') & (col('delivery_days').isNotNull())).count()

In [0]:
# Rows showing the is_delivered=yes but no delivery days
silver_orders.filter((col('is_delivered')=='yes') & (col('delivery_days').isNull())).select('*').display()

In [0]:
# Rows showing is_delivered='no but delivery days=something
silver_orders.filter((col('is_delivered')=='no') & (col('delivery_days').isNotNull())).select('*').display()

In [0]:
silver_orders.filter((col('order_approved_at').isNull()) & (col('order_status')=='delivered')).select('*').display()

## Silver Delta Table Creation and Verification

Each cleaned DataFrame is persisted with the same pattern — `.format('delta').mode('overwrite').saveAsTable(...)` — into the `workspace.olist` schema. `mode('overwrite')` keeps the notebook idempotent: rerunning it rebuilds every Silver table from the current Bronze tables.

The Silver tables created by this notebook:

| Silver table | Source Bronze table |
| --- | --- |
| `workspace.olist.silver_orders` | `workspace.default.bronze_orders` |
| `workspace.olist.silver_order_items` | `workspace.olist.bronze_order_items` |
| `workspace.olist.silver_products` | `workspace.olist.bronze_products` |
| `workspace.olist.silver_customers` | `workspace.olist.bronze_customers` |
| `workspace.olist.silver_sellers` | `workspace.olist.bronze_sellers` |

Verification after transformation:

* `DESCRIBE DETAIL` on `silver_orders` and `silver_customers` — table-level metadata (format, location, file count, size) confirming the tables were created as managed Delta tables.
* `SELECT ... LIMIT 5` on `silver_order_items` — a quick look at the persisted rows.
* A sample join of `silver_orders` and `silver_order_items` on `order_id` — a sanity check that the two tables relate correctly on their shared key.

Why these checks run after transformation: each block re-examines row counts, key uniqueness and nulls on the transformed DataFrames because the Silver layer adds derived columns (`is_delivered`, `delivery_days`, `item_value`) and rewrites identifiers (trimming, casing, ZIP padding). Re-running these checks confirms the transformations neither lost nor duplicated records and that the derived columns are consistent with the source values — the Gold aggregations depend on all of this being true.

In [0]:
# creating silver delta table
silver_orders.write.format('delta').mode('overwrite').saveAsTable('workspace.olist.silver_orders')

In [0]:
%sql

DESCRIBE DETAIL workspace.olist.silver_orders

## Order Items Transformation

The order items transformation adds a single derived column:

* **`item_value`** — `round(price + freight_value, 2)`, the total monetary value of each item row including freight. The Gold layer aggregates this column into revenue and order-value metrics, so it is computed once here, at Silver.

The notebook first explores this on `order_items_df` (reloaded from `workspace.olist.bronze_order_items` in the dependency cell at the top), then builds `silver_order_items` from the same Bronze table with the `item_value` column added.

The checks present on `silver_order_items`:

* Row count — confirms the transformation preserved record cardinality.
* Null count for `item_value` — every row must have a computable value.
* Distinct count of the composite key (`order_id`, `order_item_id`) — verifies that each item row is unique, the key the end-to-end validation notebook later relies on.

In [0]:
# Total value of each item
order_items_df=order_items_df.withColumn('item_value',round(col('price')+col('freight_value'),2))

In [0]:
order_items_df.display()

# Silver order_iems

In [0]:
# Creating silver_order_items df 
bronz_order_items=spark.table('workspace.olist.bronze_order_items')
silver_order_items=bronz_order_items.withColumn('item_value',round(col('price')+col('freight_value'),2))
silver_order_items.display()

In [0]:
silver_order_items.count()

In [0]:
silver_order_items.select(
    sum(when(col("item_value").isNull(), 1).otherwise(0))
).display()

In [0]:
silver_order_items.select(col('order_id'),col('order_item_id')).distinct().count()

In [0]:
# Saving silver_order_items as table
silver_order_items.write.mode('overwrite').saveAsTable('workspace.olist.silver_order_items')

In [0]:
%sql
select * from
workspace.olist.silver_order_items
limit 5

In [0]:
%sql
select * from
workspace.olist.silver_orders as w1
join workspace.olist.silver_order_items as w2
on w1.order_id = w2.order_id
limit 5

# Silver Product

## Products Data Transformation

The products block reads `workspace.olist.bronze_products` via `spark.read.table(...)` and works on column standardization and a data quality flag:

* **Column renaming** — `withColumnRenamed('product_name_lenght', 'product_name_length')` and `withColumnRenamed('product_description_lenght', 'product_description_length')` correct the misspelled `lenght` columns inherited from the source CSV. Note that as written these calls are not assigned back to `silver_products`, so the table persisted below keeps the original column spellings.
* **`is_weight_valid`** — `when(product_weight_g > 0, True).otherwise(False)`, flagging products whose weight is zero or negative. The Bronze profiling already surfaced zero-weight products; this column marks them explicitly. As with the renames, the result is displayed but not reassigned, so it is not part of the persisted `silver_products` table.

In [0]:
silver_products=spark.read.table('workspace.olist.bronze_products')

                                

In [0]:
silver_products.withColumnRenamed('product_name_lenght','product_name_length')
silver_products.withColumnRenamed('product_description_lenght','product_description_length')


In [0]:
silver_products.withColumn('is_weight_valid',when(
    col('product_weight_g')>0,True
).otherwise(False)).display()

In [0]:
silver_products.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.silver_products')

# Silver Customers

## Customers Data Transformation

The customers block standardizes the identifier and location columns of `workspace.olist.bronze_customers`:

* `customer_id`, `customer_unique_id` and `customer_city` — `trim(...)` removes stray whitespace.
* `customer_state` — `upper(trim(...))` puts the state codes in consistent uppercase.
* `customer_zip_code_prefix` is preserved as a **string**: a ZIP prefix such as `01001` would lose its leading zero if handled as a number, and a prefix is a categorical location code rather than a quantity — string storage keeps the value exact and avoids implying arithmetic on it.

The checks present:

* `printSchema()`, total row count, and distinct counts of `customer_id` and `customer_unique_id` — the two identifier columns have different meanings (`customer_id` is unique per order, `customer_unique_id` identifies the underlying person), so both are profiled.
* An aggregate null check on `customer_id`, `customer_unique_id` and `customer_zip_code_prefix`.

In [0]:
silver_customers = spark.table('workspace.olist.bronze_customers') \
    .withColumn('customer_id', trim(col('customer_id'))) \
    .withColumn('customer_unique_id', trim(col('customer_unique_id'))) \
    .withColumn('customer_city', trim(col('customer_city'))) \
    .withColumn('customer_state', upper(trim(col('customer_state'))))

In [0]:
silver_customers.printSchema()
silver_customers.count()
silver_customers.select('customer_id').distinct().count()
silver_customers.select('customer_unique_id').distinct().count()
silver_customers.display()

In [0]:


silver_customers.agg(
    count('*').alias('total_rows'),
    countDistinct('customer_id').alias('distinct_customer_ids'),
    countDistinct('customer_unique_id').alias('distinct_unique_customers'),
    sum(when(col('customer_id').isNull(), 1).otherwise(0)).alias('null_customer_ids'),
    sum(when(col('customer_unique_id').isNull(), 1).otherwise(0)).alias('null_unique_ids'),
    sum(when(col('customer_zip_code_prefix').isNull(), 1).otherwise(0)).alias('null_zip_codes')
).display()

In [0]:
silver_customers.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.silver_customers')

In [0]:
%sql
DESCRIBE DETAIL workspace.olist.silver_customers;

# Seller Silver

## Sellers Data Transformation

The sellers block standardizes `seller_raw` (reloaded from `workspace.olist.bronze_sellers`) into `silver_sellers` in a single `select`:

* `seller_id` — kept as the key.
* `seller_zip_code_prefix` — `lpad(cast(... as string), 5, '0')`: cast to string and zero-padded to a fixed width of 5 characters, so short prefixes keep their leading zeros and every ZIP value has a uniform format.
* `seller_city` — `lower(trim(...))`. Note the asymmetry with the customers block: seller cities are lowercased here, while customer cities are only trimmed.
* `seller_state` — `upper(trim(...))`.

The transformed DataFrame is written directly to `workspace.olist.silver_sellers` in the same cell. The verification aggregate that follows checks total rows, distinct `seller_id`, null `seller_id` and `seller_zip_code_prefix`, and `invalid_zip_lengths` — the count of ZIP prefixes whose length is not 5, which validates that the padding produced a consistent format.

In [0]:
silver_sellers = seller_raw.select(
    col('seller_id'),
    lpad(col('seller_zip_code_prefix').cast('string'), 5, '0')
        .alias('seller_zip_code_prefix'),
    lower(trim(col('seller_city'))).alias('seller_city'),
    upper(trim(col('seller_state'))).alias('seller_state')
)

silver_sellers.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.silver_sellers')

In [0]:
silver_sellers.agg(
    count('*').alias('total_rows'),
    countDistinct('seller_id').alias('distinct_sellers'),
    sum(when(col('seller_id').isNull(), 1).otherwise(0)).alias('null_seller_ids'),
    sum(when(col('seller_zip_code_prefix').isNull(), 1).otherwise(0)).alias('null_zip_codes'),
    sum(when(length(col('seller_zip_code_prefix')) != 5, 1).otherwise(0)).alias('invalid_zip_lengths')
).display()